In [ ]:
import json, os
from collections import Counter

# Load IV bigrams from Step 1
with open("/content/combined_tts_iv_clean.json", encoding="utf-8") as f:
    iv_data = json.load(f)

IV_BIGRAMS = set(iv_data["iv_bigrams"])
IV_WORDS   = set(iv_data["iv_words"])
print(f"IV bigrams : {len(IV_BIGRAMS):,}")
print(f"IV words   : {len(IV_WORDS):,}")

# Load Step 2 novel word pool
with open("/content/corpus_novel_words.json", encoding="utf-8") as f:
    step2 = json.load(f)

novel_words = step2["novel_words"]   # list of {"word": ..., "freq": ...}
print(f"Novel words loaded : {len(novel_words):,}")


IV bigrams : 2,646
IV words   : 71,844
Novel words loaded : 856,314


In [ ]:
# Step 3a: find OOV candidates and count missing-bigram frequencies

missing_bigram_freq = Counter()   # how often each missing bigram appears across the pool
oov_candidates = []               # (word, freq_in_corpus, frozenset_of_missing_bigrams)
iv_candidates  = []               # words where ALL bigrams are already in IV set

for entry in novel_words:
    word  = entry["word"]
    freq  = entry["freq"]

    if len(word) < 2:
        continue

    word_bigrams    = {word[i:i+2] for i in range(len(word) - 1)}
    missing_bigrams = word_bigrams - IV_BIGRAMS

    if missing_bigrams:
        oov_candidates.append((word, freq, frozenset(missing_bigrams)))
        for bg in missing_bigrams:
            missing_bigram_freq[bg] += 1
    else:
        iv_candidates.append((word, freq))

print(f"OOV candidates (≥1 missing bigram) : {len(oov_candidates):,}")
print(f"IV  candidates (all bigrams known)  : {len(iv_candidates):,}")
print(f"Unique missing bigrams              : {len(missing_bigram_freq):,}")
print(f"\nTop 20 most-needed missing bigrams:")
for bg, cnt in missing_bigram_freq.most_common(20):
    print(f"  {bg}  →  {cnt:,} words contain it")


OOV candidates (≥1 missing bigram) : 8,208
IV  candidates (all bigrams known)  : 848,106
Unique missing bigrams              : 962

Top 20 most-needed missing bigrams:
  ्ङ  →  130 words contain it
  इआ  →  124 words contain it
  धद  →  90 words contain it
  इख  →  88 words contain it
  जँ  →  75 words contain it
  ॉं  →  74 words contain it
  नई  →  72 words contain it
  ैे  →  72 words contain it
  िि  →  72 words contain it
  ईङ  →  70 words contain it
  डङ  →  66 words contain it
  हफ  →  63 words contain it
  ृध  →  63 words contain it
  तध  →  55 words contain it
  मउ  →  55 words contain it
  यफ  →  54 words contain it
  ोढ  →  52 words contain it
  भभ  →  52 words contain it
  धच  →  51 words contain it
  वफ  →  49 words contain it


In [ ]:
K            = 6
TARGET       = 2000
MIN_CORPUS_FREQ = 2
MAX_SHARED_BIGRAMS = 1   # skip word if ALL its missing bigrams are already being covered well

bigram_coverage = Counter()
selected        = []

ranked = sorted(
    oov_candidates,
    key=lambda x: sum(missing_bigram_freq[b] for b in x[2]),
    reverse=True
)

for word, freq, missing in ranked:
    if freq < MIN_CORPUS_FREQ:
        continue

    eligible = [b for b in missing if bigram_coverage[b] < K]
    if not eligible:
        continue

# Skip if this word only covers bigrams already seen many times
# i.e. all eligible bigrams are already covered equal or more than 3 times
    if all(bigram_coverage[b] >= 3 for b in eligible):
        continue

    selected.append({"word": word, "corpus_freq": freq,
                     "missing_bigrams": sorted(missing),
                     "n_missing": len(missing)})
    for b in eligible:
        bigram_coverage[b] += 1

    if len(selected) >= TARGET:
        break

print(f"Selected OOV words  : {len(selected):,}")
print(f"Missing bigrams covered : {len(bigram_coverage):,} / {len(missing_bigram_freq):,}")
print(f"\nSample (first 20):")
for s in selected[:20]:
    print(f"  {s['word']:25s}  freq={s['corpus_freq']:5d}  missing={s['missing_bigrams']}")

Selected OOV words  : 2,000
Missing bigrams covered : 882 / 962

Sample (first 20):
  झडङ्ङ                      freq=    5  missing=['डङ', '्ङ']
  डङ्ङगुर                    freq=    4  missing=['डङ', '्ङ']
  डङ्ङ                       freq=    3  missing=['डङ', '्ङ']
  तङ्ङु                      freq=    2  missing=['ङु', '्ङ']
  ढुङ्ङो                     freq=    2  missing=['ङो', '्ङ']
  एफइआइएमएस                  freq=    7  missing=['इआ', 'फइ']
  एफइआईएमएस                  freq=    2  missing=['इआ', 'फइ']
  एर्फइआईएमएस                freq=    2  missing=['इआ', 'फइ']
  भृ्ङ्गराज                  freq=    2  missing=['ृ्', '्ङ']
  हृवाङ्ङै                   freq=    3  missing=['ङै', '्ङ']
  ड्याङ्ङै                   freq=    2  missing=['ङै', '्ङ']
  छङ्ङ                       freq=    2  missing=['छङ', '्ङ']
  ठङ्ग्रङ्ङ                  freq=    2  missing=['ठङ', '्ङ']
  डङोलले                     freq=    2  missing=['ङो', 'डङ']
  पॉंच                       freq=   15  missing

In [ ]:
import os

os.makedirs("step3_results", exist_ok=True)

# Save selected OOV words
oov_payload = {
    "description": "Step 3 greedy OOV selection — words with missing IV bigrams, k=6 cap, target=2000",
    "k": K,
    "target": TARGET,
    "min_corpus_freq": MIN_CORPUS_FREQ,
    "n_selected": len(selected),
    "n_missing_bigrams_total": len(missing_bigram_freq),
    "n_missing_bigrams_covered": len(bigram_coverage),
    "words": selected
}

with open("step3_results/oov_selected_words.json", "w", encoding="utf-8") as f:
    json.dump(oov_payload, f, ensure_ascii=False, indent=2)

# Also save as CSV for easy review
import csv
with open("step3_results/oov_selected_words.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["word", "corpus_freq", "n_missing", "missing_bigrams"])
    writer.writeheader()
    for s in selected:
        writer.writerow({
            "word": s["word"],
            "corpus_freq": s["corpus_freq"],
            "n_missing": s["n_missing"],
            "missing_bigrams": " ".join(s["missing_bigrams"])
        })

print(f"✓ Saved → step3_results/oov_selected_words.json")
print(f"✓ Saved → step3_results/oov_selected_words.csv")
print(f"\n{'='*55}")
print("STEP 3 — SUMMARY")
print(f"{'='*55}")
print(f"  OOV candidates from Step 2 pool : {len(oov_candidates):>8,}")
print(f"  Missing bigrams found           : {len(missing_bigram_freq):>8,}")
print(f"  Missing bigrams covered         : {len(bigram_coverage):>8,}")
print(f"  Final OOV words selected        : {len(selected):>8,}")
print(f"\n  → Feed oov_selected_words.csv into Step 4 for category classification.")


✓ Saved → step3_results/oov_selected_words.json
✓ Saved → step3_results/oov_selected_words.csv

STEP 3 — SUMMARY
  OOV candidates from Step 2 pool :    8,208
  Missing bigrams found           :      962
  Missing bigrams covered         :      882
  Final OOV words selected        :    2,000

  → Feed oov_selected_words.csv into Step 4 for category classification.


In [5]:
import pandas as pd

df = pd.read_csv("step3_results/oov_selected_words.csv")
print(f"Total OOV words : {len(df):,}")
print(f"\nTop 30 by corpus frequency:")
print(df.sort_values('corpus_freq', ascending=False).head(30).to_string(index=False))
print(f"\nMissing bigrams per word:")
print(df['n_missing'].describe())


Total OOV words : 2,000

Top 30 by corpus frequency:
      word  corpus_freq  n_missing missing_bigrams
    चेन्नई          530          1              नई
    वफादार          518          1              वफ
   दिवङ्गत          388          1              वङ
    संर्घष          262          1              घष
    शाऊलले          214          1              ऊल
 उत्तरायणं          206          1              णं
   अर्धचेत          205          1              धच
सहइन्चार्ज          192          1              हइ
    दुईथरी          192          1              ईथ
  दासढुंगा          189          1              सढ
      इआइए          181          1              इआ
       यूथ          177          1              ूथ
      शाऊल          173          1              ऊल
       बीऊ          166          1              ीऊ
   साँझतिर          164          1              झत
  माघभित्र          162          1              घभ
दासढुंगामा          160          1              सढ
     सोरठी          157      